# Tapi · Exploración inicial con SQL

Ejecutá las celdas en orden. Usamos DuckDB para consultar los CSV con SQL dentro de Colab.

La celda de carga descarga los tres CSV desde GitHub y crea las tablas cuentas, consultas y pagos.


In [ ]:
%pip -q install duckdb==1.4.3

In [1]:
from pathlib import Path
from urllib.request import urlretrieve

import duckdb

url_base = "https://raw.githubusercontent.com/DaianaSep/tapi_Challenge/main"
archivos = {
    "cuentas": "B_agenda_cuentas.csv",
    "consultas": "B_agenda_consultas.csv",
    "pagos": "B_agenda_pagos.csv",
}

carpeta = Path("data")
carpeta.mkdir(exist_ok=True)

nueva_conexion = duckdb.connect()
try:
    for tabla, archivo in archivos.items():
        destino = carpeta / archivo
        urlretrieve(f"{url_base}/{archivo}", destino)
        nueva_conexion.execute(
            f"CREATE TABLE {tabla} AS "
            "SELECT * FROM read_csv(?, header=true, all_varchar=true)",
            [str(destino)],
        )

except Exception:
    nueva_conexion.close()
    raise

if "conexion" in globals():
    conexion.close()
conexion = nueva_conexion

def consultar(sql):
    return conexion.execute(sql).df()

print("CSV descargados desde GitHub. Tablas cuentas, consultas y pagos listas.")


Seleccioná los tres archivos CSV del challenge.


TypeError: 'NoneType' object is not subscriptable

Cargamos inicialmente todas las columnas como texto para conservar IDs y timestamps originales. Los campos vacíos se leen como NULL. Más adelante convertiremos fechas y montos para los cálculos que lo requieran.

## Muestra de cuentas

Cada fila representa una adhesión a la Agenda. LIMIT devuelve hasta diez filas. Sin ORDER BY no hay un orden garantizado.

In [ ]:
consultar("""
SELECT *
FROM cuentas
LIMIT 10
""")

## Muestra de consultas

Cada fila representa un intento de consulta. LIMIT devuelve hasta diez filas. Sin ORDER BY no hay un orden garantizado.

In [ ]:
consultar("""
SELECT *
FROM consultas
LIMIT 10
""")

## Muestra de pagos

Cada fila representa un pago realizado. LIMIT devuelve hasta diez filas. Sin ORDER BY no hay un orden garantizado.

In [ ]:
consultar("""
SELECT *
FROM pagos
LIMIT 10
""")

## Cantidad de registros

COUNT(*) cuenta filas. UNION ALL apila los resultados de las tres tablas.

In [ ]:
consultar("""
SELECT 'cuentas' AS tabla, COUNT(*) AS filas FROM cuentas
UNION ALL
SELECT 'consultas' AS tabla, COUNT(*) AS filas FROM consultas
UNION ALL
SELECT 'pagos' AS tabla, COUNT(*) AS filas FROM pagos
""")

## Identificadores de cuentas

Compará filas, IDs informados e IDs distintos. COUNT(columna) excluye NULL. Una diferencia requiere investigar faltantes o repeticiones.

In [ ]:
consultar("""
SELECT
    COUNT(*) AS filas,
    COUNT(account_id) AS filas_con_id,
    COUNT(DISTINCT account_id) AS cuentas_distintas
FROM cuentas
""")

## Identificadores de consultas

Compará filas, IDs informados e IDs distintos. COUNT(columna) excluye NULL. Una diferencia requiere investigar faltantes o repeticiones.

In [ ]:
consultar("""
SELECT
    COUNT(*) AS filas,
    COUNT(consulta_id) AS filas_con_id,
    COUNT(DISTINCT consulta_id) AS consultas_distintas
FROM consultas
""")

## Identificadores de pagos

Compará filas, IDs informados e IDs distintos. COUNT(columna) excluye NULL. Una diferencia requiere investigar faltantes o repeticiones.

In [ ]:
consultar("""
SELECT
    COUNT(*) AS filas,
    COUNT(pago_id) AS filas_con_id,
    COUNT(DISTINCT pago_id) AS pagos_distintos
FROM pagos
""")

## Convenios y proveedores

Una fila por combinación de convenio y proveedor. Permite identificar convenios compartidos y comparar el volumen de consultas.

In [ ]:
consultar("""
SELECT
    convenio,
    proveedor,
    COUNT(*) AS cantidad_consultas
FROM consultas
GROUP BY convenio, proveedor
ORDER BY convenio, proveedor
""")

## Tus observaciones

Anotá qué encontraste en cada consulta y qué querés investigar después. Revisaremos origen, resultado, fechas y campos vacíos en el próximo paso.

# Paso 2 · Validar los datos antes del ratio semanal

## Origen y resultado

Cada fila del resultado cuenta intentos de consulta. Una cuenta puede devolver `deuda_encontrada` varias veces.

In [ ]:
consultar("""
SELECT origen, resultado, COUNT(*) AS intentos
FROM consultas
GROUP BY origen, resultado
ORDER BY origen, intentos DESC
""")

## Fechas cubiertas

Los timestamps vienen en formato ISO. Tomamos los primeros diez caracteres para conservar el día local registrado en el CSV.

In [ ]:
consultar("""
SELECT 'cuentas' AS tabla, MIN(fecha_alta) AS desde, MAX(fecha_alta) AS hasta
FROM cuentas
UNION ALL
SELECT 'consultas', MIN(SUBSTR("timestamp", 1, 10)), MAX(SUBSTR("timestamp", 1, 10))
FROM consultas
UNION ALL
SELECT 'pagos', MIN(SUBSTR(fecha_pago, 1, 10)), MAX(SUBSTR(fecha_pago, 1, 10))
FROM pagos
""")

## Campos vacíos según resultado

En consultas sin deuda, es esperable que monto y vencimiento estén vacíos. Revisá en especial la fila de `deuda_encontrada`.

In [ ]:
consultar("""
SELECT
    resultado,
    COUNT(*) AS intentos,
    COUNT(*) FILTER (WHERE NULLIF(TRIM(monto), '') IS NULL) AS sin_monto,
    COUNT(*) FILTER (WHERE NULLIF(TRIM(fecha_vencimiento), '') IS NULL) AS sin_vencimiento
FROM consultas
GROUP BY resultado
ORDER BY resultado
""")

## Paso 3 · Ratio semanal de DeltaPay

Dividimos consultas por pagos de los convenios de DeltaPay. Las semanas van de lunes a domingo, usando la fecha local del CSV. Marcamos los extremos parciales y dejamos indefinido el ratio cuando hay cero pagos. El exceso supone que se mantienen los pagos observados.

In [ ]:
consultar("""
WITH convenios_delta AS (
    SELECT DISTINCT convenio FROM consultas WHERE proveedor = 'deltapay'
),
consultas_semanales AS (
    SELECT date_trunc('week', CAST(substr("timestamp", 1, 10) AS DATE)) AS semana,
           COUNT(*) AS consultas
    FROM consultas
    WHERE proveedor = 'deltapay'
    GROUP BY 1
),
pagos_semanales AS (
    SELECT date_trunc('week', CAST(substr(fecha_pago, 1, 10) AS DATE)) AS semana,
           COUNT(*) AS pagos
    FROM pagos
    WHERE convenio IN (SELECT convenio FROM convenios_delta)
    GROUP BY 1
)
SELECT semana, COALESCE(consultas, 0) AS consultas, COALESCE(pagos, 0) AS pagos,
       ROUND(COALESCE(consultas, 0) * 1.0 / NULLIF(pagos, 0), 2) AS consultas_por_pago,
       CASE WHEN pagos > 0 THEN GREATEST(COALESCE(consultas, 0) - 6 * pagos, 0) END AS exceso,
       semana < DATE '2026-08-01' OR semana + INTERVAL '6 days' > DATE '2026-10-31' AS semana_parcial
FROM consultas_semanales
FULL OUTER JOIN pagos_semanales USING (semana)
ORDER BY semana
""")

## Hipótesis · Guardar una deuda ya detectada

Usamos cuenta, convenio, monto y vencimiento como aproximación a una factura. Contamos las detecciones posteriores de esa combinación si no hubo un pago registrado entre ambas. Es un conteo histórico de repeticiones, no una estimación directa de ahorro. Una regla operativa necesitaría invalidar al pagar y refrescar el dato cuando pueda cambiar.

In [ ]:
consultar("""
WITH detecciones AS (
    SELECT consulta_id, account_id, convenio, monto, fecha_vencimiento, "timestamp",
           LAG("timestamp") OVER (
               PARTITION BY account_id, convenio, monto, fecha_vencimiento
               ORDER BY "timestamp", consulta_id
           ) AS deteccion_anterior
    FROM consultas
    WHERE proveedor = 'deltapay' AND resultado = 'deuda_encontrada'
      AND monto IS NOT NULL AND fecha_vencimiento IS NOT NULL
)
SELECT convenio,
       COUNT(*) AS consultas_con_deuda,
       COUNT(*) FILTER (
           WHERE deteccion_anterior IS NOT NULL
             AND NOT EXISTS (
                 SELECT 1 FROM pagos p
                 WHERE p.account_id = d.account_id AND p.convenio = d.convenio
                   AND p.monto = d.monto AND p.fecha_vencimiento = d.fecha_vencimiento
                   AND p.fecha_pago >= d.deteccion_anterior
                   AND p.fecha_pago <= d."timestamp"
             )
       ) AS detecciones_repetidas_sin_pago_intermedio
FROM detecciones d
GROUP BY convenio
ORDER BY detecciones_repetidas_sin_pago_intermedio DESC
""")